# Storm Path Visualization (LNN with Simple Autoencoder)
This notebook visualizes actual vs. predicted storm trajectories for **6h, 12h, 18h, and 24h** forecast horizons using the **Continuous-Time Liquid Neural Network (CfC Liquid U-Net) with Pretrained Simple Autoencoder**.

### Key Highlights:
1. **Sequential History Input**: Feeds expanding trajectory histories ($\ge 12$h past observation) into the continuous-time model.
2. **Feature Scaler Integration**: Loads `storm_scaler.pkl` to scale tabular inputs while preserving missing indicators (`-1.0`).
3. **Multi-Horizon Forecast**: Simultaneously outputs 4 future forecast horizons (6h, 12h, 18h, 24h) with coordinate shifts $(\Delta \text{lat}, \Delta \text{lon})$.
4. **Geospatial Rescaling & Plotting**: Rescales offsets back to true degree coordinates ($\text{Lat }^\circ\text{N}, \text{Lon }^\circ\text{E}$) and computes Great-Circle Haversine distance errors in km.

In [ ]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings('ignore')

# Add directory containing model and utils to sys.path
CURRENT_DIR = os.getcwd() if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
if CURRENT_DIR not in sys.path:
    sys.path.append(CURRENT_DIR)

from model import StormPathCfCLiquidUNet
from utils import prepare_storm_dataframe, scale_storm_features, haversine_distance_km

In [ ]:
# Paths configuration
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "..", "..", ".."))
DATA_PATH = os.path.join(PROJECT_ROOT, "Datasets", "Storm_data", "test.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(PROJECT_ROOT, "Datasets", "data.csv")

MODEL_PATH = os.path.join(CURRENT_DIR, "checkpoints", "best_storm_model.pth")
SCALER_PATH = os.path.join(CURRENT_DIR, "checkpoints", "storm_scaler.pkl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device : {device} ({torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'})")
print(f"Data Path    : {DATA_PATH}")
print(f"Model Path   : {MODEL_PATH}")
print(f"Scaler Path  : {SCALER_PATH}")

# 1. Load trained PyTorch checkpoint
checkpoint = torch.load(MODEL_PATH, map_location=device)
input_dim = checkpoint.get('input_dim', 15)
latent_dim = checkpoint.get('latent_dim', 64)
num_horizons = checkpoint.get('num_future_steps', 4)

model = StormPathCfCLiquidUNet(
    input_dim=input_dim,
    latent_dim=latent_dim,
    unet_hidden_dims=(64, 128, 256),
    fc_hidden_dim=128,
    num_horizons=num_horizons,
    nodes_per_horizon=2,
    dropout=0.0
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"[SUCCESS] Loaded model from epoch {checkpoint.get('epoch', 'N/A')}")

# 2. Load fitted scaler
with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)
print(f"[SUCCESS] Loaded {type(scaler).__name__} with {len(getattr(scaler, 'feature_names_in_', []))} features")

In [ ]:
# 3. Load Dataset & Perform Feature Engineering and Scaling
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
if 'time' in df_raw.columns:
    df_raw['time'] = pd.to_datetime(df_raw['time'])
    if df_raw['time'].dt.year.min() < 1980:
        df_raw = df_raw[df_raw['time'].dt.year >= 1980].copy()

df_clean, feature_cols = prepare_storm_dataframe(df_raw)

# Scale tabular features using the fitted scaler
df_scaled, _ = scale_storm_features(df_clean, feature_cols, scaler=scaler, fit_scaler=False)

# Extract a sample storm for visualization
sample_storm_id = df_clean['international_id'].unique()[0]
storm_data = df_clean[df_clean['international_id'] == sample_storm_id].copy().reset_index(drop=True)
storm_scaled = df_scaled.loc[df_clean['international_id'] == sample_storm_id].copy().reset_index(drop=True)

print(f"Selected Storm ID  : {sample_storm_id}")
print(f"Total Observations : {len(storm_data)}")
print(f"Feature Columns ({len(feature_cols)}): {feature_cols}")

In [ ]:
import random

def plot_lnn_storm_prediction(
    storm_data,
    storm_scaled,
    model,
    feature_cols,
    idx=None,
    device=device,
    margin=5.0
):
    """
    Visualizes actual vs. predicted storm paths for LNN with Simple Autoencoder.
    
    Parameters
    ----------
    storm_data : pd.DataFrame
        Raw unscaled DataFrame of the storm containing 'lat', 'lon', 'year', 'month', 'day', 'hour'.
    storm_scaled : pd.DataFrame
        Fitted-scaled features DataFrame for the storm.
    model : nn.Module
        Trained StormPathCfCLiquidUNet model.
    feature_cols : List[str]
        List of feature names expected by the model.
    idx : Optional[int]
        Index of observation point to forecast from. If None, chosen randomly with >= 12h history.
    margin : float
        Geographic degree margin around track for the map extent.
    """
    min_history = 2
    num_future_steps = 4
    n_states = len(storm_data)
    
    if n_states < min_history + 1:
        print(f"Storm is too short ({n_states} points) for trajectory prediction.")
        return
        
    if idx is None:
        max_idx = max(min_history, n_states - num_future_steps)
        idx = random.randint(min_history - 1, max(min_history - 1, max_idx))
        
    idx = max(min_history - 1, min(idx, n_states - 1))
    
    # 1. Extract historical features sequence (0 to idx inclusive)
    x_seq = storm_scaled.iloc[0 : idx + 1][feature_cols].values
    x_tensor = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(0).to(device)
    
    # 2. Run model forward pass
    with torch.no_grad():
        preds = model(x_tensor).cpu().numpy()[0]  # Shape: (4, 2) [[dlat_6h, dlon_6h], ...]
        
    # 3. Extract current location in true geographic coordinates (degrees)
    current_lat = float(storm_data.iloc[idx]['lat'])
    current_lon = float(storm_data.iloc[idx]['lon'])
    current_year = int(storm_data.iloc[idx]['year'])
    current_month = int(storm_data.iloc[idx]['month'])
    current_day = int(storm_data.iloc[idx]['day'])
    current_hour = int(storm_data.iloc[idx]['hour'])
    time_str = f"{current_year}-{current_month:02d}-{current_day:02d} {current_hour:02d}:00 UTC"
    
    # 4. Rescale predictions: Add displacements to current coordinate
    horizons = ['6h', '12h', '18h', '24h'][:num_future_steps]
    colors = {'6h': 'orange', '12h': 'red', '18h': 'purple', '24h': 'black'}
    
    pred_lats = [current_lat]
    pred_lons = [current_lon]
    
    for h_idx in range(len(horizons)):
        delta_lat = float(preds[h_idx, 0])
        delta_lon = float(preds[h_idx, 1])
        pred_lats.append(current_lat + delta_lat)
        pred_lons.append(current_lon + delta_lon)
        
    # 5. Extract ground truth actual coordinates if available
    actual_future_lats = [current_lat]
    actual_future_lons = [current_lon]
    dist_errors_km = {}
    
    for step in range(1, num_future_steps + 1):
        if idx + step < n_states:
            act_lat = float(storm_data.iloc[idx + step]['lat'])
            act_lon = float(storm_data.iloc[idx + step]['lon'])
            actual_future_lats.append(act_lat)
            actual_future_lons.append(act_lon)
            
            p_lat = pred_lats[step]
            p_lon = pred_lons[step]
            err_km = haversine_distance_km(np.array([act_lat]), np.array([act_lon]), np.array([p_lat]), np.array([p_lon]))[0]
            dist_errors_km[horizons[step - 1]] = err_km
            
    # 6. Render Geospatial Map using Cartopy
    plt.figure(figsize=(13, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.coastlines(resolution='50m', linewidth=1.0)
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.7)
    ax.add_feature(cfeature.LAND, edgecolor='black', facecolor='#E8E8E8', zorder=1)
    ax.add_feature(cfeature.OCEAN, facecolor='#D8EAF5', zorder=1)
    
    # Focused map extent
    lon_min = min(min(pred_lons), min(storm_data.iloc[0:idx+1]['lon'])) - margin
    lon_max = max(max(pred_lons), max(storm_data.iloc[0:idx+1]['lon'])) + margin
    lat_min = min(min(pred_lats), min(storm_data.iloc[0:idx+1]['lat'])) - margin
    lat_max = max(max(pred_lats), max(storm_data.iloc[0:idx+1]['lat'])) + margin
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    # Plot full actual storm trajectory (reference background)
    ax.plot(storm_data['lon'], storm_data['lat'], marker='o', markersize=3, color='gray', alpha=0.5, label='Actual Full Path', transform=ccrs.PlateCarree(), zorder=2)
    
    # Plot past observed history
    past_lons = storm_data.iloc[0 : idx + 1]['lon'].values
    past_lats = storm_data.iloc[0 : idx + 1]['lat'].values
    ax.plot(past_lons, past_lats, color='blue', linewidth=2.2, alpha=0.8, label=f'Observed History ({(idx+1)*6}h)', transform=ccrs.PlateCarree(), zorder=3)
    
    # Plot ground truth future path
    if len(actual_future_lats) > 1:
        ax.plot(actual_future_lons, actual_future_lats, color='green', linestyle='--', linewidth=2.2, label='Actual Future Track', transform=ccrs.PlateCarree(), zorder=4)
        ax.scatter(actual_future_lons[1:], actual_future_lats[1:], color='green', marker='o', s=55, zorder=5, transform=ccrs.PlateCarree())
        
    # Plot current position
    ax.plot(current_lon, current_lat, marker='*', markersize=22, color='yellow', markeredgecolor='red', markeredgewidth=2, label=f'Current Position (t={idx})', transform=ccrs.PlateCarree(), zorder=6)
    
    # Plot predicted path and horizon markers
    for i, h in enumerate(horizons):
        err_str = f' ({dist_errors_km[h]:.1f} km)' if h in dist_errors_km else ''
        ax.plot(pred_lons[i+1], pred_lats[i+1], marker='X', markersize=11, color=colors[h], linestyle='None', label=f'LNN Forecast {h}{err_str}', transform=ccrs.PlateCarree(), zorder=5)
        
    ax.plot(pred_lons, pred_lats, color='red', linestyle='--', linewidth=2.2, alpha=0.85, label='Predicted Trajectory', transform=ccrs.PlateCarree(), zorder=4)
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linestyle=':', color='gray', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    
    storm_id_val = storm_data['international_id'].iloc[0]
    plt.title(f'Storm #{storm_id_val} Path Prediction (LNN + Simple Autoencoder)\nForecast from {time_str} [Lat {current_lat:.2f}°, Lon {current_lon:.2f}°]', fontsize=13, fontweight='bold')
    plt.legend(loc='lower left', framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()
    
    # Print forecast accuracy summary
    if dist_errors_km:
        print("\n" + "=" * 52)
        print("            FORECAST ACCURACY SUMMARY")
        print("=" * 52)
        for h, err in dist_errors_km.items():
            print(f"  Forecast Horizon {h:>4}: Distance Error = {err:6.2f} km")
        print("=" * 52)

In [ ]:
# Run the visualization on the selected storm at observation step index 5 (36h past observation)
plot_lnn_storm_prediction(storm_data, storm_scaled, model, feature_cols, idx=5, margin=6.0)

In [ ]:
# Multi-Storm / Trajectory Inspection: Test another storm from the dataset
unique_storms = df_clean['international_id'].unique()
if len(unique_storms) > 1:
    another_storm_id = unique_storms[1]
    another_storm_data = df_clean[df_clean['international_id'] == another_storm_id].copy().reset_index(drop=True)
    another_storm_scaled = df_scaled.loc[df_clean['international_id'] == another_storm_id].copy().reset_index(drop=True)
    print(f"Visualizing Storm #{another_storm_id} (Total states: {len(another_storm_data)})...")
    plot_lnn_storm_prediction(another_storm_data, another_storm_scaled, model, feature_cols, idx=None, margin=6.0)
else:
    print("Re-running on storm with another random observation index:")
    plot_lnn_storm_prediction(storm_data, storm_scaled, model, feature_cols, idx=None, margin=6.0)